### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [ ]:
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [ ]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [ ]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [ ]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
MetaData=(
    pl.scan_parquet(folder / 'MetaData.parquet')
            .filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))
)
# MetaData.row(0, named=True) 
MetaData.collect()

In [ ]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')

There are only time climate zones for the data

----------------------------------------------

### Prepare cross-validation model


In [ ]:
df.collect_schema()

In [ ]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).cast(pl.UInt32).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.cooling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()

In [ ]:
x.head(6)

In [ ]:
y.head(6)

In [ ]:
# confirming types
x.schema

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    # return mod.score(x_test, y_test)
    # return r2_score(y_test, predicted)
    # return mean_absolute_error(y_test, predicted)
    # return mean_squared_error(y_test, predicted)
    return root_mean_squared_error(y_test, predicted)

In [ ]:
%%time
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
skf= StratifiedKFold(n_splits=6)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    LR=model(LinearRegression(), x_train, x_test, y_train, y_test)
    RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test)
    XG=model(xg.XGBRegressor(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Linear Regression": [LR],
    "Random Forest": [RF],
    "XG Boost": [XG]
    })
    arr.append(frame)
lf=pd.concat(arr)

## Calculating measures

In [ ]:
lf.mean()

In [ ]:
## cross val score
LR=LinearRegression()
RF=RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1)
XG=xg.XGBRegressor()

In [ ]:
np.mean(cross_val_score(LR, x, y, cv=10))

In [ ]:
np.mean(cross_val_score(RF, x, y, cv=10))

In [ ]:
np.mean(cross_val_score(XG, x, y, cv=10))

In [ ]:
lf.mean()

## preprocess that data for better accuracy